In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, random_split, DataLoader
from torchvision import datasets, transforms
import torch.nn.functional as F
import torch.optim as optim

import matplotlib.pyplot as plt

import os
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [ ]:
class BottleneckLayer(nn.Module):
    def __init__(self, in_channels, filters, strides=1):
        super(BottleneckLayer, self).__init__()
        self.skip_connection = None

        #TODO: Fill in the ___ lines below, so that the first convolutional layer outputs 4 times the number of starting filters/channels.
        self.conv1 = nn.Conv2d(in_channels,filters*4, kernel_size=1, stride=strides, padding='same')
        self.bn1 = nn.BatchNorm2d(filters*4)
        self.relu1 = nn.ReLU()
    
        #TODO: Fill in the ___ lines below, so that the second convolutional layer accepts 4 times the number of starting filters/channels 
        # and outputs the original number of starting filters. 
        self.conv2 = nn.Conv2d(filters*4,filters, kernel_size=3, stride=strides, padding='same')
        self.bn2 = nn.BatchNorm2d(filters) 
        self.relu2 = nn.ReLU()

    def forward(self, x):
        # save an initial version of the input for skip connection later on
        self.skip_connection = x

        x = self.bn1(self.conv1(x))
        x = self.relu1(x)

        #TODO: Run the second batch normalization, convolution, and ReLU layer! 
        x = self.bn2(self.conv2(x))
        x = self.relu2(x)

        # Skip connection step
        x = torch.cat((x, self.skip_connection), dim=1)

        return x

In [ ]:
class TransitionLayer(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.bn = nn.BatchNorm2d(in_channels)
        self.relu = nn.ReLU()
        
        #TODO: Create a convolutional layer that shrinks the number of channels by half. 
        self.conv = nn.Conv2d(
            in_channels,
            in_channels//2,
            kernel_size=1,
            stride=1,
            padding=0  
        )
        
        #TODO: Create a pooling layer that shrinks the dimensions of the feature maps by half.  
        self.pool = nn.AvgPool2d(kernel_size=2, stride=2, padding=0)
        self.out_channels = in_channels // 2

    def forward(self, x):
        x = self.bn(x)
        x = self.relu(x)
        
        #TODO: Run the convolution and pooling layer!
        x = self.conv(x)
        x = self.pool(x)
        return x

In [ ]:
class Denseblock(nn.Module):
    def __init__(self, in_channels, repetition=1, growth_rate=32):
        super(DenseBlock, self).__init__()

        #store in a ModuleList, not a regular list so that PyTorch recognizes that this layer is part of the model
        #otherwise, it wouldn't show in model parameters or get moved to the GPU...
        self.layers = nn.ModuleList()

        #need to keep track of the number of input channels, this grows as we go through the block
        channels = in_channels

        for _ in range(repetition):
            #TODO: Append the output of a BottleneckLayer with the appropriate number of channels and new filters. 
            self.layers.append(BottleneckLayer(channels, growth_rate))
            
            # each bottleneck concatenates its output (growth_rate channels)
            # onto its input, so the next layer sees more input channels
            channels += growth_rate
        self.out_channels = channels  
    
        for layer in self.layers:
            x = layer(x)
        return x

In [2]:
# Creating DenseNet-B121
class DenseNetB121(nn.Module):
    def __init__(self, input_shape, num_classes, growth_rate=32):
        """
        input_shape: (channels, height, width) -- PyTorch uses channels-first,
            
        """
        super().__init__()
        in_channels = input_shape[0]

        # Initial layer
        #TODO: Fill out all of the blanks!
        self.bn_init = nn.BatchNorm2d(in_channels)
        self.relu_init = nn.ReLU()
        self.conv_init = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3)
        # Pooling layer
        self.pool_init = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        channels = 64

        # First dense and transition layer
        self.dense1 = DenseBlock(channels, 6, growth_rate)
        channels = self.dense1.out_channels
        self.trans1 = TransitionLayer(channels)
        channels = self.trans1.out_channels

        #TODO: Write up the code for the second dense and transition layer, third dense and transition layer, and last dense layer
        #Second dense and transition layer
        self.dense2 = DenseBlock(channels,12, growth_rate)
        channels = self.dense2.out_channels
        self.trans2 = TransitionLayer(channels)
        channels = self.trans2.out_channels

        #Third dense and transition layer
        self.dense3 = DenseBlock(channels, 24, growth_rate)
        channels = self.dense3.out_channels
        self.trans3 = transitionLayer(channels)
        channels = self.tran3.out_channels

        #Last dense layer
        self.dense4 = DenseBlock(channels,16)
        channels = self.dense4.out_channels
       
    

        # Global average pooling + output layer
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        #TODO: Add the final fully connected layer to output the disease class
        self.fc = nn.linear(channels,num_classes)

    
    def forward(self, x):
        x = self.bn_init(x)
        x = self.relu_init(x)
        x = self.conv_init(x)
        x = self.pool_init(x)

        x = self.dense1(x)
        x = self.trans1(x)

        #TODO: Run the second dense and transition layer, third dense and transition layer, and last dense layer
        x = self.dense2(x)
        x = self.trans2(x)

        x = self.dense3(x)
        x = self.trans3(x)

        x = self.dense4(x)


        x = self.global_pool(x)          # shape: (batch, channels, 1, 1)
        x = torch.flatten(x, 1)          # shape: (batch, channels)
        x = self.fc(x)                   # logits, shape: (batch, num_classes)
        return x                         # softmax applied via loss fn, see note below

NameError: name 'nn' is not defined

In [ ]:
IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
IMAGE_CHANNELS = 3

#TODO: Find the directory that contains all of our training images. Hint: use os.listdir to explore file paths in "/kaggle/input".
train_dir = os.path.join("/kaggle/input", os.listdir("/kaggle/input")[0])
# transforms: resize to match TF's image_size behavior, then convert to tensor
transform = transforms.Compose([
    transforms.Resize((IMAGE_HEIGHT, IMAGE_WIDTH)),
    transforms.ToTensor(),  # scales pixel values to [0, 1] and reorders to (C, H, W)
])

# Load the full dataset (ImageFolder infers classes from subdirectory names,
# same as image_dataset_from_directory does)
full_dataset = datasets.ImageFolder(train_dir, transform=transform)

# Compute the 80/20 split sizes
val_size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size

# Seed must match exactly between subsets -- here that just means using
# the same generator seed once, since random_split does both splits in one call
generator = torch.Generator().manual_seed(123)
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

In [ ]:
NUM_CLASSES = 10

class EncodedDataset(Dataset):
    def __init__(self, dataset, num_classes=NUM_CLASSES):
        self.dataset = dataset
        self.num_classes = num_classes

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        features, label = self.dataset[idx]
        # One-hot encode the label
        label = F.one_hot(torch.tensor(label), num_classes=self.num_classes).float()
        return features, label


encoded_train_ds = EncodedDataset(train_dataset, NUM_CLASSES)
encoded_val_ds = EncodedDataset(val_dataset, NUM_CLASSES)

# Wrap in DataLoaders as before
train_ds = DataLoader(encoded_train_ds, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_ds = DataLoader(encoded_val_ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

In [1]:
import time
device = torch.device('cuda')

# Construct the model, passing in_channels manually (as your comment notes)
# TODO: Write the appropriate dimensions and number of classes. Hint: there are already global variables defined for these values!
model = DenseNetB121(input_shape=(IMAGE_CHANNELS, IMAGE_HEIGHT, IMAGE_WIDTH), num_classes=len(full_dataset.classes))
model = model.to(device)

# "Compile" step: optimizer + loss
optimizer = optim.Adam(model.parameters(), eps=0.05)
loss_fn = nn.CrossEntropyLoss()  

EPOCHS = 1
history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS} started...")
    epoch_start = time.time()

    # --- Training phase ---
    model.train()
    running_loss, running_correct, running_total = 0.0, 0, 0

    for features, labels in train_ds:
        features, labels = features.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(features)          # raw logits
        loss = loss_fn(outputs, labels)    # labels are one-hot, CrossEntropyLoss handles this
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * features.size(0)
        preds = outputs.argmax(dim=1)
        true_labels = labels.argmax(dim=1)  # convert one-hot back to class index for accuracy
        running_correct += (preds == true_labels).sum().item()
        running_total += features.size(0)

    #TODO: Compute training loss and accuracy by dividing the running loss and accuracy by the total number of features:
        train_loss = running_loss / running_total
        train_acc = running_correct / running_total
    # --- Validation phase ---
    model.eval()
    val_running_loss, val_running_correct, val_running_total = 0.0, 0, 0

    with torch.no_grad():
        for features, labels in val_ds:
            features, labels = features.to(device), labels.to(device)

            outputs = model(features)
            loss = loss_fn(outputs, labels)

            val_running_loss += loss.item() * features.size(0)
            preds = outputs.argmax(dim=1)
            true_labels = labels.argmax(dim=1)
            val_running_correct += (preds == true_labels).sum().item()
            val_running_total += features.size(0)

    #TODO: Compute validation loss and accuracy by dividing the running loss and accuracy by the total number of features: 
            val_loss = val_running_loss / val_running_total
            val_acc = val_running_correct / val_running_total

    history['loss'].append(train_loss)
    history['accuracy'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)

    epoch_time = time.time() - epoch_start
    print(f"Epoch {epoch+1}/{EPOCHS} finished in {epoch_time:.1f}s")
    print(f"  loss: {train_loss:.4f} - accuracy: {train_acc:.4f} - "
          f"val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")

NameError: name 'torch' is not defined

In [3]:
import os

output_dir = '/kaggle/working/'
os.makedirs(output_dir, exist_ok=True)

# Save just the model weights (recommended — smaller file, more portable)
model_path = os.path.join(output_dir, 'densenetb121_paddy.pth')
torch.save(model.state_dict(), model_path)
print(f"Model weights saved to {model_path}")

# Optionally, also save the full training history for later analysis/plotting
import json
history_path = os.path.join(output_dir, 'history.json')
with open(history_path, 'w') as f:
    json.dump(history, f)
print(f"Training history saved to {history_path}")

NameError: name 'torch' is not defined